# 01 — Source Profiling (Workshop #1)
**Team 6 — Volcanic Activity Monitoring** · Data Analysis Programming 2026-II · Universidad Distrital

This notebook profiles the **real, unmodified** responses stored in `workshop_1/samples/` by `scripts/fetch_samples.py`:

| # | Source | Role | Format |
|---|---|---|---|
| H1 | Smithsonian GVP — Holocene volcanoes + eruptions (WFS) | historical | GeoJSON |
| H2 | USGS FDSN event service, radial queries around volcanoes | historical | CSV |
| I1 | Smithsonian / USGS Weekly Volcanic Activity Report | incremental | RSS XML |
| I2 | USGS real-time feed `all_week` | incremental | GeoJSON |

Sections: 1) inventory · 2) per-source profiles (nulls, ranges, duplicates, time zones, coordinates, units) · 3) update behaviour between two polls · 4) data-quality problems · 5) cross-source overlap (candidate same-event pairs) · 6) volume estimate.

In [ ]:
import json, os, re, sys, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from volcano_pipeline.sources import weekly_report, gvp
from volcano_pipeline.normalization.geo import haversine_km, valid_coordinates

SAMPLES = Path(os.getenv("SAMPLES_DIR", ROOT / "workshop_1" / "samples"))
RAW = Path(os.getenv("RAW_DATA_DIR", ROOT / "data" / "raw"))
RADIUS_KM = 30
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("samples dir:", SAMPLES)

## 1. Inventory of the evidence files

In [ ]:
rows = []
for f in sorted(SAMPLES.glob("*")):
    if f.name.endswith(".meta.json") or f.is_dir():
        continue
    meta_f = f.with_suffix(f.suffix + ".meta.json")
    meta = json.loads(meta_f.read_text()) if meta_f.exists() else {}
    rows.append({"file": f.name, "source_id": meta.get("source_id", f.name.split("_2")[0]),
                 "retrieved_utc": meta.get("retrieved_utc"), "KB": round(f.stat().st_size/1024, 1),
                 "sha256": (meta.get("sha256") or "")[:12],
                 "etag": meta.get("headers", {}).get("ETag") or meta.get("headers", {}).get("Etag"),
                 "last_modified": meta.get("headers", {}).get("Last-Modified")})
inventory = pd.DataFrame(rows)
inventory

In [ ]:
def files(prefix, ext=None):
    fs = [f for f in sorted(SAMPLES.glob(prefix + "*")) if not f.name.endswith(".meta.json")]
    return [f for f in fs if ext is None or f.suffix == ext]

def latest(prefix):
    fs = files(prefix)
    if not fs:
        raise FileNotFoundError(f"No sample for {prefix}: run python scripts/fetch_samples.py")
    return fs[-1]

def profile(df, name):
    """Generic profile: dtype, null %, distinct, min/max per column."""
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_%": (df.isna().mean() * 100).round(2),
        "distinct": df.nunique(dropna=True),
    })
    num = df.select_dtypes("number")
    out["min"] = num.min(); out["max"] = num.max()
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns")
    return out

def col(df, *cands):
    for c in cands:
        if c in df.columns:
            return c
    raise KeyError(f"none of {cands} in columns")

## 2. Source profiles
### 2.1 H1 — GVP Holocene volcanoes (static reference entity)

In [ ]:
vol_file = latest("gvp_volcanoes")
volcanoes = pd.DataFrame(gvp.parse_features(vol_file.read_bytes()))
print("file:", vol_file.name)
print("columns:", list(volcanoes.columns))
profile(volcanoes, "GVP volcanoes")

In [ ]:
VN, VNAME = col(volcanoes, "Volcano_Number"), col(volcanoes, "Volcano_Name")
VLAT, VLON = col(volcanoes, "Latitude", "_lat"), col(volcanoes, "Longitude", "_lon")
volcanoes[VN] = pd.to_numeric(volcanoes[VN], errors="coerce").astype("Int64")
print("duplicated volcano numbers:", volcanoes[VN].duplicated().sum())
print("invalid coordinates:", (~volcanoes.apply(lambda r: valid_coordinates(r[VLAT], r[VLON]), axis=1)).sum())
lec = col(volcanoes, "Last_Eruption_Year")
ley = pd.to_numeric(volcanoes[lec], errors="coerce")
print(f"Last_Eruption_Year: non-numeric/unknown = {ley.isna().sum()}, BCE (<0) = {(ley < 0).sum()}, since 1900 = {(ley >= 1900).sum()}")
volcanoes[[VN, VNAME, col(volcanoes, 'Country'), VLAT, VLON, lec]].head()

### 2.2 H1 — GVP Holocene eruptions

In [ ]:
er_file = latest("gvp_eruptions")
eruptions = pd.DataFrame(gvp.parse_features(er_file.read_bytes()))
print("file:", er_file.name); print("columns:", list(eruptions.columns))
profile(eruptions, "GVP eruptions")

In [ ]:
EN = col(eruptions, "Eruption_Number"); EVN = col(eruptions, "Volcano_Number")
VEI = col(eruptions, "ExplosivityIndexMax", "VEI")
SY, SM, SD = col(eruptions, "StartDateYear"), col(eruptions, "StartDateMonth"), col(eruptions, "StartDateDay")
for c in (SY, SM, SD, VEI, EVN):
    eruptions[c] = pd.to_numeric(eruptions[c], errors="coerce")
print("duplicated Eruption_Number:", eruptions[EN].duplicated().sum())
print(f"VEI missing: {eruptions[VEI].isna().mean():.1%}  range: {eruptions[VEI].min()}-{eruptions[VEI].max()}")
precision = np.select([eruptions[SD].fillna(0) > 0, eruptions[SM].fillna(0) > 0], ["day", "month"], "year")
print("start-date precision:\n", pd.Series(precision).value_counts())
orphans = ~eruptions[EVN].isin(volcanoes[VN].astype(float))
print("eruptions whose volcano is not in the Holocene volcano list:", orphans.sum())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
eruptions[VEI].value_counts().sort_index().plot.bar(ax=ax[0], title="VEI distribution")
eruptions.loc[eruptions[SY] >= 1960, SY].plot.hist(bins=40, ax=ax[1], title="Eruption start year (>=1960)")
plt.tight_layout()

### 2.3 H2 — USGS FDSN earthquakes around volcanoes (CSV)

In [ ]:
fdsn_files = files("usgs_fdsn_", ".csv")
parts = []
for f in fdsn_files:
    d = pd.read_csv(f)
    m = re.search(r"_v(\d{6})_", f.name)
    d["query_vnum"] = int(m.group(1)) if m else None
    parts.append(d)
fdsn = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print(len(fdsn_files), "files; columns:", list(fdsn.columns))
profile(fdsn, "USGS FDSN")

In [ ]:
print("time zone check (all times end with Z):", fdsn["time"].astype(str).str.endswith("Z").all())
fdsn["time_utc"] = pd.to_datetime(fdsn["time"], utc=True, format="ISO8601")
fdsn["updated_utc"] = pd.to_datetime(fdsn["updated"], utc=True, format="ISO8601")
print("coverage:", fdsn["time_utc"].min(), "->", fdsn["time_utc"].max())
print("rows per query volcano:\n", fdsn["query_vnum"].value_counts())
print("\nevent types:", fdsn["type"].value_counts().to_dict())
print("magnitude types:", fdsn["magType"].value_counts().to_dict())
print("status:", fdsn["status"].value_counts().to_dict())
print("negative depths (above sea level):", (fdsn["depth"] < 0).sum())
dup_ids = fdsn["id"].duplicated(keep=False)
print("ids repeated across volcano queries (one quake near 2+ volcanoes):", fdsn.loc[dup_ids, "id"].nunique())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
fdsn["mag"].plot.hist(bins=40, ax=ax[0], title="Magnitude")
fdsn.set_index("time_utc").resample("YE")["id"].count().plot.bar(ax=ax[1], title="Events per year")
plt.tight_layout()

### 2.4 I1 — Weekly Volcanic Activity Report (RSS)

In [ ]:
rss_files = files("gvp_weekly_rss", ".xml")
weekly = pd.DataFrame(weekly_report.parse_rss(rss_files[-1].read_bytes()))
print("file:", rss_files[-1].name)
display(profile(weekly.drop(columns=["description"]), "Weekly report"))
print("title parse OK:", weekly["title_parse_ok"].mean())
print("status:", weekly["status"].value_counts().to_dict())
print("vnum found in link:", weekly["volcano_number"].notna().mean())
print("vnum present in GVP list:", weekly["volcano_number"].isin(volcanoes[VN].astype(float)).mean())
print("description length (chars):", weekly["description"].str.len().describe()[["min","50%","max"]].to_dict())
weekly[["volcano_name", "country", "week_start", "week_end", "status", "volcano_number", "latitude", "longitude", "pub_date_utc"]]

### 2.5 I2 — USGS real-time GeoJSON feed (`all_week`)

In [ ]:
feed_files = files("usgs_feed", ".geojson")
def load_feed(f):
    d = json.loads(f.read_bytes())
    rows = []
    for ft in d["features"]:
        p = ft["properties"]; lon, lat, depth = ft["geometry"]["coordinates"]
        rows.append({"id": ft["id"], "time_ms": p["time"], "updated_ms": p["updated"], "mag": p["mag"],
                     "magType": p.get("magType"), "type": p.get("type"), "status": p.get("status"),
                     "place": p.get("place"), "net": p.get("net"), "tsunami": p.get("tsunami"),
                     "lat": lat, "lon": lon, "depth": depth})
    return d["metadata"], pd.DataFrame(rows)
feed_meta, feed = load_feed(feed_files[-1])
print("metadata:", {k: feed_meta.get(k) for k in ("generated", "count", "title", "api")})
feed["time_utc"] = pd.to_datetime(feed["time_ms"], unit="ms", utc=True)
feed["updated_utc"] = pd.to_datetime(feed["updated_ms"], unit="ms", utc=True)
display(profile(feed, "USGS feed all_week"))
print("types:", feed["type"].value_counts().to_dict())
print("magTypes:", feed["magType"].value_counts().head(10).to_dict())
print("null magnitude:", feed["mag"].isna().sum(), " duplicated ids:", feed["id"].duplicated().sum())

In [ ]:
# Which feed earthquakes are within RADIUS_KM of a Holocene volcano?
vlat = volcanoes[VLAT].astype(float).to_numpy(); vlon = volcanoes[VLON].astype(float).to_numpy()
def nearest_volcano(lat, lon):
    la1, lo1, la2, lo2 = map(np.radians, (lat, lon, vlat, vlon))
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    d = 2*6371.0088*np.arcsin(np.sqrt(a)); i = int(np.nanargmin(d))
    return volcanoes[VN].iloc[i], volcanoes[VNAME].iloc[i], float(d[i])
feed[["near_vnum", "near_name", "near_km"]] = feed.apply(lambda r: pd.Series(nearest_volcano(r.lat, r.lon)), axis=1)
near = feed[feed["near_km"] <= RADIUS_KM]
print(f"{len(near)} of {len(feed)} feed events ({len(near)/len(feed):.1%}) are within {RADIUS_KM} km of a volcano")
near.groupby("near_name").size().sort_values(ascending=False).head(15)

## 3. Update behaviour (two polls ≥ 1 hour apart)
`scripts/fetch_samples.py --poll` stores a new snapshot. Here the two most recent snapshots of each incremental source are compared: new ids, ids that disappeared, and records whose `updated`/content changed.

In [ ]:
if len(feed_files) >= 2:
    m0, f0 = load_feed(feed_files[-2]); m1, f1 = load_feed(feed_files[-1])
    t0 = pd.to_datetime(m0["generated"], unit="ms", utc=True); t1 = pd.to_datetime(m1["generated"], unit="ms", utc=True)
    print(f"poll A {t0}  vs  poll B {t1}  (gap {t1 - t0})")
    a = f0.drop_duplicates("id", keep="last").set_index("id")
    b = f1.drop_duplicates("id", keep="last").set_index("id")
    new_ids, gone_ids = b.index.difference(a.index), a.index.difference(b.index)
    common = a.index.intersection(b.index)
    changed = common[(a.loc[common, "updated_ms"] != b.loc[common, "updated_ms"]).to_numpy()]
    mag_changed = common[(a.loc[common, "mag"] != b.loc[common, "mag"]).to_numpy()]
    print(f"new: {len(new_ids)}  disappeared: {len(gone_ids)} (aged out of 7-day window or deleted)  "
          f"updated changed: {len(changed)}  magnitude revised: {len(mag_changed)}")
    wm = a["updated_ms"].max()
    print("records with updated > watermark of poll A:", (b["updated_ms"] > wm).sum(),
          "(= new + changed: the incremental rule works)")
    display(pd.concat([a.loc[mag_changed, ["mag", "magType", "status", "updated_ms"]].add_suffix("_A"),
                       b.loc[mag_changed, ["mag", "magType", "status", "updated_ms"]].add_suffix("_B")], axis=1).head(10))
else:
    print("Only one feed snapshot: run `python scripts/fetch_samples.py --poll` at least 1 hour later.")

In [ ]:
if len(rss_files) >= 2:
    w0 = pd.DataFrame(weekly_report.parse_rss(rss_files[-2].read_bytes()))
    w1 = pd.DataFrame(weekly_report.parse_rss(rss_files[-1].read_bytes()))
    k = lambda d: set(zip(*d.dropna(subset=["volcano_number", "week_start"])[["volcano_number", "week_start"]].T.values))
    print("weeks in poll A:", sorted(w0.week_start.dropna().unique()), " poll B:", sorted(w1.week_start.dropna().unique()))
    print("new (vnum, week) keys:", len(k(w1) - k(w0)), " removed:", len(k(w0) - k(w1)))
    same = w0.merge(w1, on=["volcano_number", "week_start"], suffixes=("_A", "_B"))
    print("same key but edited text:", (same.description_A != same.description_B).sum())
else:
    print("Only one RSS snapshot: poll again after the next Wednesday 23:00 UTC publication.")

## 4. Data-quality problems found (automatically evidenced)
Each finding prints an example record. The narrative and the handling strategy are in the report.

In [ ]:
findings = []
def finding(title, example, strategy):
    findings.append({"problem": title, "example": example, "planned handling": strategy})

x = eruptions[eruptions[VEI].isna()].head(1)
if len(x): finding(f"Missing VEI in {eruptions[VEI].isna().mean():.0%} of eruptions",
                   x[[EN, EVN, SY, VEI]].to_dict("records")[0], "keep NULL severity; flag; exclude from VEI statistics")
x = eruptions[(eruptions[SM].fillna(0) == 0)].head(1)
if len(x): finding("Partial eruption dates (month/day = 0 or null)", x[[EN, SY, SM, SD]].to_dict("records")[0],
                   "first day of period + time_precision column")
x = eruptions[eruptions[SY] < 1].head(1)
if len(x): finding("BCE (negative) years cannot be timestamps", x[[EN, SY]].to_dict("records")[0],
                   "event_time NULL, keep start_year; exclude from time series")
x = volcanoes[pd.to_numeric(volcanoes[lec], errors="coerce").isna()].head(1)
if len(x): finding("Non-numeric Last_Eruption_Year (e.g. 'Unknown')", x[[VN, VNAME, lec]].to_dict("records")[0],
                   "map to NULL + evidence category")
if fdsn["magType"].nunique() > 1:
    finding(f"Mixed magnitude types in one catalog ({fdsn['magType'].nunique()} types)",
            fdsn.groupby("magType").head(1)[["id", "mag", "magType"]].head(3).to_dict("records"),
            "store magType; never compare magnitudes of different types without labeling")
x = fdsn[fdsn["depth"] < 0].head(1)
if len(x): finding("Negative depths (hypocentre above sea level)", x[["id", "depth", "place"]].to_dict("records")[0],
                   "valid at volcanoes: accept range [-5, 800] km")
x = fdsn[fdsn["type"] != "earthquake"].head(1)
if len(x): finding("Non-earthquake events in catalog (explosion, quarry blast, ...)",
                   x[["id", "type", "place"]].to_dict("records")[0], "reject to quarantine with reason 'non_tectonic_type'")
x = fdsn[fdsn["id"].duplicated(keep=False)].sort_values("id").head(2)
if len(x): finding("Same earthquake returned by two volcano queries", x[["id", "query_vnum"]].to_dict("records"),
                   "dedupe on id; keep both links in volcano_earthquake")
for c in ("magError", "depthError", "horizontalError", "nst", "gap", "dmin", "rms"):
    if c in fdsn and fdsn[c].isna().mean() > 0.2:
        finding(f"High null rate in FDSN quality field '{c}' ({fdsn[c].isna().mean():.0%})",
                fdsn[fdsn[c].isna()][["id", c]].head(1).to_dict("records")[0], "optional field: keep NULL, not used in rules")
        break
x = feed[feed["mag"].isna()].head(1)
if len(x): finding("Feed events without magnitude", x[["id", "mag", "status"]].to_dict("records")[0],
                   "accept with NULL severity; exclude from magnitude indicators")
x = weekly[weekly["volcano_number"].isna() | ~weekly["title_parse_ok"]].head(1)
if len(x): finding("Weekly item without parseable vnum/title", x[["title", "link"]].to_dict("records")[0],
                   "name match fallback; else quarantine 'unlinked_report'")
x = weekly[weekly["volcano_name"].str.contains(",", na=False)].head(1)
if len(x): finding("Inverted names with commas ('Ruiz, Nevado del') -> naive name joins fail",
                   x[["volcano_name", "volcano_number"]].to_dict("records")[0], "join by vnum, never by name")
finding("Four different timestamp encodings across sources",
        {"usgs_feed": "epoch ms", "usgs_fdsn": "ISO-8601 Z", "rss": "RFC-822 pubDate + text week range",
         "gvp": "separate year/month/day ints"}, "normalize everything to timestamptz UTC")
dq = pd.DataFrame(findings); print(len(dq), "problems found"); dq

## 5. Cross-source overlap — candidate same-event pairs
Two kinds of overlap matter for this hazard:
1. **Same earthquake in two sources** (FDSN historical vs real-time feed): same `id`; differences in `mag`, `status`, `updated`.
2. **Same volcanic episode in two sources** (weekly report vs GVP eruption record): same `vnum`, report week inside the eruption interval (eruption ongoing or ended after the week start).

In [ ]:
pairs = fdsn.merge(feed, on="id", suffixes=("_fdsn", "_feed"))
pairs["mag_diff"] = pairs["mag_feed"] - pairs["mag_fdsn"]
pairs["updated_diff"] = pairs["updated_utc_feed"] - pairs["updated_utc_fdsn"]
print("earthquakes present in both FDSN sample and feed:", len(pairs))
pairs[["id", "time_utc_fdsn", "mag_fdsn", "mag_feed", "magType_fdsn", "status_fdsn", "status_feed", "updated_diff", "query_vnum"]].head(8)

In [ ]:
ey = col(eruptions, "EndDateYear")
eruptions[ey] = pd.to_numeric(eruptions[ey], errors="coerce")
recent_er = eruptions[eruptions[SY] >= 2000]
cand = weekly.merge(recent_er, left_on="volcano_number", right_on=EVN, how="inner")
cand["week_year"] = pd.to_datetime(cand["week_start"]).dt.year
cand = cand[(cand[ey].isna()) | (cand[ey] >= cand["week_year"])]           # ongoing or ended after report
cand = cand.sort_values(SY).groupby("volcano_number").tail(1)
cand["gvp_start"] = cand[SY].astype("Int64").astype(str) + "-" + cand[SM].fillna(0).astype(int).astype(str).str.zfill(2) \
                    + "-" + cand[SD].fillna(0).astype(int).astype(str).str.zfill(2)
print("weekly-report items matched to an open GVP eruption:", len(cand), "of", len(weekly))
cand[["volcano_name", "volcano_number", "week_start", "status", EN, "gvp_start", ey, VEI]].head(10)

In [ ]:
# Earthquakes in the feed near volcanoes that are in this week's report (the join the indicator relies on)
wk_v = set(weekly["volcano_number"].dropna().astype(int))
near_reported = near[near["near_vnum"].astype(int).isin(wk_v)]
print(f"{len(near_reported)} feed earthquakes within {RADIUS_KM} km of volcanoes reported this week")
near_reported[["id", "time_utc", "mag", "magType", "near_name", "near_km"]].sort_values("time_utc").tail(10)

## 6. Volume estimate for the bulk load

In [ ]:
cnt_files = files("usgs_fdsn_counts", ".json")
gvp_full = sorted(RAW.glob("gvp_eruptions_full_*.geojson"))
est = []
est.append({"source": "gvp_volcanoes", "rows": len(volcanoes), "MB": vol_file.stat().st_size/1e6})
if gvp_full:
    est.append({"source": "gvp_eruptions (full)", "rows": len(json.loads(gvp_full[-1].read_bytes())["features"]),
                "MB": gvp_full[-1].stat().st_size/1e6})
bytes_per_row = sum(f.stat().st_size for f in fdsn_files) / max(len(fdsn), 1)
if cnt_files:
    c = json.loads(cnt_files[-1].read_text())
    total = sum(c["counts"].values())
    est.append({"source": f"usgs_fdsn ({len(c['counts'])} volcanoes, {c['start']}..{c['end']})", "rows": total,
                "MB": total * bytes_per_row / 1e6})
else:
    per_v = fdsn.groupby("query_vnum").size().median()
    n_sel = int((pd.to_numeric(volcanoes[lec], errors="coerce") >= 1900).sum())
    est.append({"source": f"usgs_fdsn (extrapolated: median {per_v:.0f}/volcano x {n_sel})",
                "rows": int(per_v * n_sel), "MB": per_v * n_sel * bytes_per_row / 1e6})
est.append({"source": "usgs_feed all_week (per poll)", "rows": len(feed), "MB": feed_files[-1].stat().st_size/1e6})
est.append({"source": "weekly report (per week)", "rows": len(weekly), "MB": rss_files[-1].stat().st_size/1e6})
pd.DataFrame(est).round(2)

**Conclusions** (fill in after running with real data): sources validated, main quality issues, matching keys confirmed (`usgs:<id>` for earthquakes, `vnum` for volcano-level linking), and bulk-load partitioning (FDSN per volcano, split by year only when `count` > 20,000).